# Notebook 04 – Điểm tín dụng, hạn mức BNPL và giải thích mô hình

1. Hiệu chỉnh xác suất vỡ nợ (PD)
2. Quy đổi PD → điểm tín dụng → hạng rủi ro
3. Chính sách duyệt và hạn mức tự động
4. Phân tích lợi nhuận theo ngưỡng duyệt
5. Độ ổn định theo thời gian (Gini theo tuần, PSI)
6. Giải thích bằng SHAP

Ngưỡng và hạng rủi ro được xác định trên tập **valid**; tập **test** chỉ dùng để báo cáo.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt, seaborn as sns
import utils
from utils import META

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
print("Du lieu:", DATA_DIR); print("Ket qua:", OUT_DIR)

import joblib
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

va = pd.read_parquet(OUT_DIR / "preds_valid.parquet")
te = pd.read_parquet(OUT_DIR / "preds_test.parquet")
tr = pd.read_parquet(OUT_DIR / "preds_train.parquet")

## 1. Hiệu chỉnh xác suất (isotonic, fit trên valid)

In [ ]:
iso = IsotonicRegression(y_min=1e-4, y_max=1 - 1e-4, out_of_bounds="clip").fit(va["pd_raw"], va["target"])
for d in (va, te): d["pd"] = iso.predict(d["pd_raw"])

print("Test - truoc hieu chinh:", {k: round(v, 4) for k, v in utils.evaluate(te.target, te.pd_raw).items()})
print("Test - sau hieu chinh  :", {k: round(v, 4) for k, v in utils.evaluate(te.target, te.pd).items()})
print("PD trung binh du doan %.2f%% | ty le vo no thuc te %.2f%%" % (100 * te.pd.mean(), 100 * te.target.mean()))

fig, ax = plt.subplots(figsize=(5, 4.5))
for col, lab in [("pd_raw", "truoc"), ("pd", "sau")]:
    y, x = calibration_curve(te["target"], te[col], n_bins=10, strategy="quantile"); ax.plot(x, y, "o-", label=lab)
m = max(ax.get_xlim()[1], ax.get_ylim()[1]); ax.plot([0, m], [0, m], "k--")
ax.set_xlabel("PD du doan"); ax.set_ylabel("Ty le vo no thuc te"); ax.set_title("Calibration (test)"); ax.legend(); plt.show()

## 2. Điểm tín dụng và hạng rủi ro
`score = OFFSET + FACTOR × ln(odds)` với odds = (1 − PD) / PD. Quy ước: 600 điểm tại odds 50:1, cứ +20 điểm thì odds gấp đôi (PDO = 20).

In [ ]:
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
FACTOR = PDO / np.log(2); OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)
for d in (va, te): d["score"] = OFFSET + FACTOR * np.log((1 - d["pd"]) / d["pd"])

BANDS = ["E", "D", "C", "B", "A"]                       # E = rui ro cao nhat
edges = np.quantile(va["score"], [0.10, 0.30, 0.60, 0.85])   # xac dinh tren valid
edges = np.concatenate([[-np.inf], np.unique(edges), [np.inf]])
labels = BANDS[-(len(edges) - 1):]
for d in (va, te): d["band"] = pd.cut(d["score"], edges, labels=labels)

band_tbl = te.groupby("band", observed=True).agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"),
                                                 pd_tb=("pd", "mean"), diem_min=("score", "min"), diem_max=("score", "max"))
band_tbl["ty_trong"] = band_tbl["so_ho_so"] / band_tbl["so_ho_so"].sum()
band_tbl.round(4)

> Kiểm tra: tỷ lệ vỡ nợ phải **giảm dần** từ hạng E đến hạng A trên tập test. Nếu không đơn điệu, gộp hạng hoặc đổi điểm cắt.

## 3. Chính sách duyệt và hạn mức

Các tham số dưới đây là **giả định nghiệp vụ** để minh hoạ (không phải dữ liệu) – cần nêu rõ trong báo cáo và có thể điều chỉnh.

In [ ]:
POLICY = pd.DataFrame({
    "band":      ["A", "B", "C", "D", "E"],
    "quyet_dinh": ["Duyet", "Duyet", "Duyet", "Duyet han che", "Tu choi"],
    "han_muc":   [20_000_000, 12_000_000, 6_000_000, 2_000_000, 0],   # VND, gia dinh
}).set_index("band")
te["han_muc"] = te["band"].astype(str).map(POLICY["han_muc"]).fillna(0)
te["quyet_dinh"] = te["band"].astype(str).map(POLICY["quyet_dinh"])
display(POLICY)
print("Ty le duyet: %.1f%%" % (100 * (te.han_muc > 0).mean()))
print("Ty le vo no trong nhom duoc duyet: %.2f%% (toan bo: %.2f%%)" % (100 * te.loc[te.han_muc > 0, "target"].mean(), 100 * te.target.mean()))

## 4. Lợi nhuận theo chính sách hạn mức và theo ngưỡng duyệt
Giả định nghiệp vụ: khoản không vỡ nợ mang lại `MARGIN` × hạn mức; khoản vỡ nợ mất `LGD` × hạn mức (dư nợ tại thời điểm vỡ nợ = hạn mức, giả định thận trọng).
- **4a.** Lợi nhuận của chính sách ở mục 3, tính đúng hạn mức của từng hạng.
- **4b.** Đường cong lợi nhuận theo ngưỡng PD: hồ sơ được duyệt nhận hạn mức của hạng mình (hạng E nếu được duyệt nhận hạn mức thấp nhất 2 triệu). Ngưỡng tối ưu chọn trên valid.
- **4c.** Phân tích độ nhạy: cùng đường cong nhưng mọi hồ sơ cùng hạn mức 6 triệu (cách tính của phiên bản trước) – chỉ để so sánh.

In [ ]:
MARGIN, LGD = 0.06, 0.60          # gia dinh nghiep vu
LIM_IF_APPROVED = POLICY["han_muc"].replace(0, POLICY.loc[POLICY["han_muc"] > 0, "han_muc"].min())  # E -> 2tr neu duoc duyet

def pnl(target, limit):
    return np.where(np.asarray(target) == 0, MARGIN, -LGD) * np.asarray(limit, dtype=float)

# ---- 4a. Chinh sach theo hang (test)
te["pnl"] = pnl(te["target"], te["han_muc"])
te["el"] = te["pd"] * LGD * te["han_muc"]                     # ton that ky vong theo PD da hieu chinh
pol = te.groupby("band", observed=True).agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"),
                                            han_muc=("han_muc", "first"), tong_han_muc=("han_muc", "sum"),
                                            ton_that_ky_vong=("el", "sum"), loi_nhuan=("pnl", "sum"))
pol.loc["Tong"] = [pol["so_ho_so"].sum(), te["target"].mean(), np.nan, pol["tong_han_muc"].sum(),
                   pol["ton_that_ky_vong"].sum(), pol["loi_nhuan"].sum()]
for c in ["tong_han_muc", "ton_that_ky_vong", "loi_nhuan"]: pol[c] = pol[c] / 1e9     # ty VND
display(pol.round(4))
print("Loi nhuan chinh sach theo hang (test): %.1f ty VND tren tong han muc %.1f ty" % (pol.loc["Tong", "loi_nhuan"], pol.loc["Tong", "tong_han_muc"]))

# ---- 4b. Duong cong theo nguong PD, han muc theo hang
def profit_curve(d, limit=None):
    lim = (d["band"].astype(str).map(LIM_IF_APPROVED).astype(float) if limit is None
           else pd.Series(float(limit), index=d.index))
    rows = []
    for t in np.quantile(d["pd"], np.linspace(0.05, 1, 40)):
        ok = d["pd"] <= t
        rows.append({"nguong_pd": t, "ty_le_duyet": ok.mean(), "ty_le_vo_no": d.loc[ok, "target"].mean(),
                     "loi_nhuan": pnl(d.loc[ok, "target"], lim[ok]).sum()})
    return pd.DataFrame(rows)

def report(pc_va, d, limit=None, label=""):
    t = pc_va.loc[pc_va["loi_nhuan"].idxmax(), "nguong_pd"]         # chon tren valid
    lim = (d["band"].astype(str).map(LIM_IF_APPROVED).astype(float) if limit is None
           else pd.Series(float(limit), index=d.index))
    ok = d["pd"] <= t
    print("%s nguong PD toi uu (valid) %.3f -> test: duyet %.1f%% | vo no %.2f%% | loi nhuan %.1f ty (duyet tat ca: %.1f ty)" % (
        label, t, 100 * ok.mean(), 100 * d.loc[ok, "target"].mean(),
        pnl(d.loc[ok, "target"], lim[ok]).sum() / 1e9, pnl(d["target"], lim).sum() / 1e9))
    return t

pc_va, pc_te = profit_curve(va), profit_curve(te)
best_t = report(pc_va, te, label="[Han muc theo hang]")

# ---- 4c. Do nhay: han muc co dinh 6tr
pc_va6, pc_te6 = profit_curve(va, 6_000_000), profit_curve(te, 6_000_000)
report(pc_va6, te, 6_000_000, label="[Han muc co dinh 6tr]")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(100 * pc_te["ty_le_duyet"], pc_te["loi_nhuan"] / 1e9, "o-", label="han muc theo hang")
ax.plot(100 * pc_te6["ty_le_duyet"], pc_te6["loi_nhuan"] / 1e9, "s--", label="co dinh 6 trieu (do nhay)")
ax.set_xlabel("Ty le duyet (%)"); ax.set_ylabel("Loi nhuan (ty VND)"); ax.set_title("Loi nhuan theo ty le duyet (test)")
ax.legend(); plt.show()

## 5. Độ ổn định theo thời gian

In [ ]:
gw = utils.gini_by_week(te.assign(score=te["pd"]))
print("Gini theo tuan: tb %.3f | min %.3f | do lech chuan %.3f" % (gw.mean(), gw.min(), gw.std()))
gw.plot(marker="o", figsize=(8, 3), title="Gini theo tuan (test)"); plt.axhline(gw.mean(), ls="--", c="gray"); plt.show()
print("PSI diem du doan train -> test: %.3f  (>0.25 la troi manh)" % utils.psi(tr["pd_raw"], te["pd_raw"]))

## 6. Giải thích mô hình bằng SHAP

In [ ]:
import shap
pack = joblib.load(OUT_DIR / "best_model.joblib"); model, feats = pack["model"], pack["features"]
df = pl.read_parquet(OUT_DIR / "features_v1.parquet").to_pandas()
_, _, test_full = utils.time_split(df)
Xs = test_full.sample(min(len(test_full), 5000), random_state=42)

explainer = shap.TreeExplainer(model.booster_)
sv = explainer.shap_values(Xs[feats])
sv = sv[1] if isinstance(sv, list) else sv
shap.summary_plot(sv, Xs[feats], max_display=20)

In [ ]:
# Ly do tu choi cho mot ho so rui ro cao: 5 dac trung day PD len nhieu nhat
i = int(np.argmax(sv.sum(axis=1)))
reason = pd.DataFrame({"gia_tri": Xs[feats].iloc[i].astype(object).values, "shap": sv[i]}, index=feats)
print("case_id:", Xs["case_id"].iloc[i], "| target thuc te:", Xs["target"].iloc[i])
reason.sort_values("shap", ascending=False).head(5)

## 7. Kết luận (tự viết)
- Mô hình chọn, chỉ số trên test, mức ổn định theo thời gian: ...
- Chính sách hạn mức đề xuất và tác động (tỷ lệ duyệt, tỷ lệ vỡ nợ, lợi nhuận): ...
- Hạn chế: không có dữ liệu hành vi duyệt web; tham số lợi nhuận là giả định; ...